In [1]:
import pandas as pd

df = pd.read_csv('finance_reddit_posts.csv')
df.head(5) 

,post_id,title,score,comments,author,created,source_endpoint
0,1tf2ppl,¿Qué pasa después del breakout del Opening Ran...,0,5,mariodoblep,5/16/2026 19:11,new
1,1text64,Do I freak the fuck out and sell it all?,66,126,ZaraBloom418,5/16/2026 16:06,new
2,1texmf0,Is the SOFI hype warranted?,38,57,ZTB1313,5/16/2026 16:00,new
3,1texhaz,"So uhh, when do I get the forbidden phone (mar...",31,35,DeltaVx_,5/16/2026 15:54,new
4,1tex6av,"China, US agree to reduce tariffs on unspecifi...",581,119,Force_Hammer,5/16/2026 15:43,new


In [2]:
# Check dimensions and column names

print(df.shape)
print(df.columns.tolist())

# post_id appears to be the unique identifier

(13748, 7)
['post_id', 'title', 'score', 'comments', 'author', 'created', 'source_endpoint']


In [3]:
# checking for duplicates

df.duplicated(subset=['post_id']).sum()

0

In [4]:
#checking for null values
df.isnull().sum()

post_id            0
title              0
score              0
comments           0
author             0
created            0
source_endpoint    0
dtype: int64

In [5]:
df.info()
df.describe()

#score & commets are important columns -> indicate engagement 


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13748 entries, 0 to 13747
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   post_id          13748 non-null  object
 1   title            13748 non-null  object
 2   score            13748 non-null  int64 
 3   comments         13748 non-null  int64 
 4   author           13748 non-null  object
 5   created          13748 non-null  object
 6   source_endpoint  13748 non-null  object
dtypes: int64(2), object(5)
memory usage: 752.0+ KB


,score,comments
count,13748.000000,13748.000000
mean,776.837504,159.107361
std,3337.737761,695.357893
min,0.000000,0.000000
25%,3.000000,8.000000
50%,32.000000,33.000000
75%,283.000000,122.000000
max,126928.000000,14787.000000


In [6]:
# Getting columns ready for analysis

import pandas as pd

FILE_NAME = r"c:\Users\Usuario\Desktop\code\Reddit_project\finance_reddit_posts.csv"

# Load data
df = pd.read_csv(FILE_NAME, engine="python", on_bad_lines="skip")

# Convert created column to datetime
df["created"] = pd.to_datetime(df["created"], format="mixed", errors="coerce")

# Clean source_endpoint text
df["source_endpoint"] = df["source_endpoint"].astype(str).str.strip()

# Known feed names
feeds = ["top_month", "top_year", "top_week", "top_day", "rising", "hot", "new", "top"]

def get_subreddit_and_feed(source):
    # get subreddit & feed
    for feed in feeds:
        if source.endswith("_" + feed):
            subreddit = source.replace("_" + feed, "")
            return subreddit, feed

    # Old format like old and hot
    if source in feeds:
        return "unknown", source

    
    if source == "wsb":
        return "wallstreetbets", "unknown"

    # if the source is the subreddit name
    return source, "unknown"

# Creating subreddit and feed columns
df[["subreddit", "feed"]] = df["source_endpoint"].apply(lambda source: pd.Series(get_subreddit_and_feed(source)))


df[["source_endpoint", "subreddit", "feed", "created"]].head(20)

#we know some of the feed is new, but we dont know from which subreddit they come from 

,source_endpoint,subreddit,feed,created
0,new,unknown,new,2026-05-16 19:11:00
1,new,unknown,new,2026-05-16 16:06:00
2,new,unknown,new,2026-05-16 16:00:00
3,new,unknown,new,2026-05-16 15:54:00
4,new,unknown,new,2026-05-16 15:43:00
5,new,unknown,new,2026-05-16 15:36:00
6,new,unknown,new,2026-05-16 13:40:00
7,new,unknown,new,2026-05-16 13:38:00
8,new,unknown,new,2026-05-16 13:35:00
9,new,unknown,new,2026-05-16 13:33:00


In [7]:
df[["source_endpoint", "subreddit", "feed", "created"]].tail(20) #new data has a clear subreddit and feed

,source_endpoint,subreddit,feed,created
13728,pennystocks_top_year,pennystocks,top_year,2025-09-27 02:37:00
13729,pennystocks_top_year,pennystocks,top_year,2025-10-22 16:35:00
13730,pennystocks_top_year,pennystocks,top_year,2025-10-06 12:01:00
13731,pennystocks_top_year,pennystocks,top_year,2025-09-25 17:59:00
13732,pennystocks_top_year,pennystocks,top_year,2025-06-30 22:44:00
13733,pennystocks_top_year,pennystocks,top_year,2025-10-24 04:01:00
13734,pennystocks_top_year,pennystocks,top_year,2025-10-15 11:40:00
13735,pennystocks_top_year,pennystocks,top_year,2025-09-26 11:24:00
13736,pennystocks_top_year,pennystocks,top_year,2025-07-29 04:01:00
13737,pennystocks_top_year,pennystocks,top_year,2025-07-14 04:04:00


In [8]:
#SPAM/BOT FILETERING

print(df['author'].value_counts().head(20))

# droping suspicious users

bots = ['AutoModerator', 'wsbapp', '[deleted]'] #list of potential bots

df = df[~df['author'].isin(bots)] # drops them 

author
AutoModerator          284
[deleted]              197
callsonreddit          187
Beren-                 168
joe4942                103
Illustrious_Lie_954     83
Force_Hammer            74
Electrical_Top_9933     71
Any_Pomegranate1134     64
Front-Page_News         60
Every-Actuator-6996     55
wsbapp                  49
Puginator               47
investorinvestor        44
JuniorCharge4571        41
app1310                 39
tandroide               30
WickedSensitiveCrew     29
raytoei                 27
TACO_Orange_3098        26
Name: count, dtype: int64


In [9]:
#saving clean data

df.to_csv('posts_clean.csv', index=False)